In [15]:
import pandas as pd

file_path = "/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang.csv"

df = pd.read_csv(file_path)

print(df.head())

   rank                            artist                    title    region  \
0     1        Mr Plata, El Americano 4KT           Las Muñequitas  Colombia   
1     2            ARIA VEGA, Ryan Castro  CHÉVERE (premium_remix)  Colombia   
2     3        Ryan Castro, Kapo, Gangsta                 LA VILLA  Colombia   
3     4                           Kris R.                    GANAS  Colombia   
4     5  W Sound, Beéle, Ovy On The Drums    La Plena - W Sound 05  Colombia   

              spotify_uri                                             lyrics  \
0  4nJJCRYru4QQakCiUA155f  (Dímelo, ¿me vas a dar lo que yo pido?)\nDame ...   
1  3CBEVPwR3kUXDoTx1lqFUQ  ARIA VEGA, Ryan Castro\nLa costeñita premium y...   
2  2ZyrAym0sRLwt4PhGotHuI  Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...   
3  4KE9Ne3hgh18B3Th4xcylg  Yeah, yeah\nYeah, yeah\n\nMi amor, culeemos co...   
4  6iOndD4OFo7GkaDypWQIou  O-O-Ovy On The Drums\nYou're the apple of my e...   

  language  
0       es  
1       es  

In [16]:
import duckdb as dd
con = dd.connect()
lang_groups = con.execute("SELECT language, COUNT(*) AS count FROM df GROUP BY language order by language").df()

display(lang_groups)

,language,count
0,ar,2
1,de,3
2,en,340
3,es,211
4,fr,3
5,gd,2
6,he,1
7,id,3
8,it,3
9,ja,4


In [17]:
df_lyrics_lang = con.execute("""
            SELECT spotify_uri, left(lyrics, 100) AS lyrics, language
            FROM df 
            where 1=1
            and language is not null
            and language not in ('en', 'unknown')
            order by language
            """).df()

display(df_lyrics_lang)

,spotify_uri,lyrics,language
0,54PbBpquVfhfrwRwvjSXbI,لم نعد نتحدث، لم نعد نتحدث\nلم نعد نتحدث كما ك...,ar
1,54PbBpquVfhfrwRwvjSXbI,لم نعد نتحدث، لم نعد نتحدث\nلم نعد نتحدث كما ك...,ar
2,2gam98EZKrF9XuOkU13ApN,Lenk' ich dich ab?\nNein\nDachte ich mir\n\nWi...,de
3,5rb9QrpfcKFHM1EUbSIurX,Peace! Und das Zeichen für die Stadt mit A\nJa...,de
4,3B54sVLJ402zGa6Xm4YGNe,Es ist nicht gut genug für mich\nSeitdem ich m...,de
...,...,...,...
346,5TkQbhQm9BXONk2agDo4w9,還沒好好的感受 雪花綻放的氣候\n我們一起顫抖 會更明白什麼是溫柔\n還沒跟你牽著手 走過荒...,zh
347,2fc2vUA5H3mTAr9rQoBDKt,抓不住愛情的我、總是眼睜睜看她溜走\n世界上幸福的人到處有、為何不能算我一個\n為了愛孤軍奮...,zh
348,4OoExItZJ0jePoCZDbHx4t,你的回話凌亂著 在這個時刻\n我想起噴泉旁的白鴿 甜蜜散落了\n情緒莫名的拉扯 我還愛你呢\...,zh
349,73Ijz3vN9KP1W5qrfNeWMI,我知 妳老母都嫌我醜\n我知 妳老爸都看我不爽快\n我知 妳家的狗 咬我三次\n全家伙都賭爛...,zh


In [13]:
from pathlib import Path
import pandas as pd
from deep_translator import GoogleTranslator

LANGUAGE_CODE_MAP = {
    "zh-cn": "zh-CN",
    "zh_cn": "zh-CN",
    "zh-tw": "zh-TW",
    "zh_tw": "zh-TW",
    "jp": "ja",
    "kr": "ko",
    "latin": "auto",
    "other": "auto",
    "unknown": "auto",
}

translator_cache = {}

def normalize_source_language(language_value):
    if pd.isna(language_value):
        return "auto"

    language = str(language_value).strip().lower()
    if not language:
        return "auto"

    return LANGUAGE_CODE_MAP.get(language, language)

def get_translator(source_language: str):
    if source_language not in translator_cache:
        translator_cache[source_language] = GoogleTranslator(source=source_language, target="en")
    return translator_cache[source_language]

def translate_to_english(text: str, source_language="auto"):
    if pd.isna(text):
        return pd.NA

    text = str(text).strip()
    if not text:
        return pd.NA

    normalized_source = normalize_source_language(source_language)

    try:
        return get_translator(normalized_source).translate(text)
    except Exception:
        if normalized_source != "auto":
            try:
                return get_translator("auto").translate(text)
            except Exception:
                return pd.NA
        return pd.NA

In [ ]:
# Translate all non-English rows; keep English rows as null in lyrics_en
df_translated = df.copy()

if "lyrics" not in df_translated.columns or "language" not in df_translated.columns:
    raise KeyError("Expected columns 'lyrics' and 'language' in the input CSV")

output_path = Path(file_path).with_name("lyrics_lang_trans.csv")
checkpoint_every = 10
cjk_pattern = r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]"

# Resume support: continue from prior partial output if present
if output_path.exists():
    existing = pd.read_csv(output_path)
    if "lyrics_en" in existing.columns and len(existing) == len(df_translated):
        df_translated["lyrics_en"] = existing["lyrics_en"]
    else:
        df_translated["lyrics_en"] = pd.NA
else:
    df_translated["lyrics_en"] = pd.NA

non_english_mask = df_translated["language"].fillna("").str.lower() != "en"
has_lyrics_mask = df_translated["lyrics"].notna()
still_cjk_mask = df_translated["lyrics_en"].fillna("").str.contains(cjk_pattern, regex=True)
needs_translation_mask = non_english_mask & has_lyrics_mask & (
    df_translated["lyrics_en"].isna() | still_cjk_mask
)

pending_indices = df_translated.index[needs_translation_mask].tolist()
total_pending = len(pending_indices)
print(f"Pending translations: {total_pending:,}")

since_last_save = 0
processed = 0
for idx in pending_indices:
    df_translated.at[idx, "lyrics_en"] = translate_to_english(
        df_translated.at[idx, "lyrics"],
        source_language=df_translated.at[idx, "language"],
    )
    since_last_save += 1
    processed += 1

    if since_last_save >= checkpoint_every:
        df_translated.to_csv(output_path, index=False)
        print(f"Progress: {processed}/{total_pending} (checkpoint saved)")
        since_last_save = 0

# Ensure English rows remain null by design
english_mask = df_translated["language"].fillna("").str.lower() == "en"
df_translated.loc[english_mask, "lyrics_en"] = pd.NA

# Final save for any remaining rows
df_translated.to_csv(output_path, index=False)

print(f"Progress: {processed}/{total_pending} (final save)")
print(f"Wrote {len(df_translated):,} rows to {output_path}")

display(df_translated[["spotify_uri", "language", "lyrics_en"]].head(10))

Pending translations: 351
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Checkpoint saved to /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03

,spotify_uri,language,lyrics_en
0,4nJJCRYru4QQakCiUA155f,es,"(Tell me, are you going to give me what I ask ..."
1,3CBEVPwR3kUXDoTx1lqFUQ,es,"ARIA VEGA, Ryan Castro\nThe premium costñita a..."
2,2ZyrAym0sRLwt4PhGotHuI,es,"Kapo, Ryan Castro, Gangsta\nWhat a joke, SOG\n..."
3,4KE9Ne3hgh18B3Th4xcylg,es,"Yeah, yeah\nYeah, yeah\n\nMy love, let's fuck ..."
4,6iOndD4OFo7GkaDypWQIou,en,<NA>
5,1HEwEN64NjgTaHmo7LfkX8,es,"(Uh-wu-wu-wu-wu-wu-wu-wu!)\nBaby, what are you..."
6,7mU1fei7P9h4mpjP2Otdw5,es,You can't say rude things\nI am Godzilla and I...
7,6VfL3MEuYeJbDlD8m011HR,es,How do I explain to my heart that you want to ...
8,5yXt80BNZGbmHFd0NHZHNn,es,I know the last thing you want now is to know ...
9,2E4TYekUduml1DWIqQWNcj,es,"And sing to him beautifully, sir!\n\nI know th..."
